# Notebook A: YOLOv8l Training for VisDrone Object Detection
**Project:** UCU AI 2026 — Multiple Object Video Detection (HW2)

**Purpose:** 
This notebook trains a YOLOv8l detector on the VisDrone2019-VID dataset. It is the first of a two-notebook pipeline:
- **Notebook A (this one):** Train YOLOv8l at `imgsz=800` for 48 epochs with SGD + cosine LR, W&B logging, and automatic checkpointing every 4 epochs.
- **Notebook B (next):** Apply SAHI (Slicing Aided Hyper Inference) for small-object detection at test time and generate the final Kaggle submission.

**Key configuration:**
- **Model:** YOLOv8l (COCO-pretrained, ~43.7M params)
- **Image size:** 800×800 (higher than default 640 for small objects)
- **Epochs:** 48
- **Optimizer:** SGD, `lr0=0.01`, cosine schedule
- **Checkpointing:** Every 4 epochs (`save_period=4`) for crash resilience
- **Tracking:** Weights & Biases (`visdrone-det` project, run `yolov8l_run1`)

**Why YOLOv8l?**
VisDrone has a very high proportion of small objects (24.9% of objects are <32×32 px). Literature shows YOLOv8l + SAHI reaches **39.30% mAP@0.5** on VisDrone, which is a significant improvement over our current Faster R-CNN baseline (val mAP = 0.1561, public LB = 0.18888).

In [1]:
# =============================================================================
# Cell 1 (Final Fix): Setup — Separate COMP_DIR (images) and DATASET_DIR (common.py)
# =============================================================================

!pip install -q ultralytics sahi

import os
import sys
import shutil
from pathlib import Path
import yaml
from ultralytics import YOLO

# ---- 1. Define both paths -------------------------------------------------
# Images and competition data
COMP_DIR = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
# common.py and weights
DATASET_DIR = Path("/kaggle/input/datasets/suchok/my-dataset")

WORK_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORK_DIR / "yolo_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- 2. Validate paths ----------------------------------------------------
if not COMP_DIR.exists():
    raise FileNotFoundError(f"❌ Competition folder missing: {COMP_DIR}")
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"❌ Dataset folder (common.py) missing: {DATASET_DIR}")

# Check for common.py
if not (DATASET_DIR / "common.py").exists():
    print(f"⚠️ Warning: common.py not found in {DATASET_DIR}. This is fine for YOLO training, but SAHI might need it later.")

# Check for train/val images
train_imgs = COMP_DIR / "train" / "images"
val_imgs   = COMP_DIR / "val"   / "images"

if not train_imgs.exists():
    raise FileNotFoundError(f"❌ Train images missing: {train_imgs}")
if not val_imgs.exists():
    raise FileNotFoundError(f"❌ Val images missing: {val_imgs}")

n_train = len(list(train_imgs.rglob("*.jpg")))
n_val   = len(list(val_imgs.rglob("*.jpg")))
print(f"✅ train frames: {n_train:,}")
print(f"✅ val frames:   {n_val:,}")

# Add DATASET_DIR to sys.path for later use (SAHI, etc.)
sys.path.insert(0, str(DATASET_DIR))

# ---- 3. Generate data.yaml pointing to COMP_DIR ---------------------------
yaml_content = {
    "path":  str(COMP_DIR),           # <--- Points to competition data
    "train": "train/images",
    "val":   "val/images",
    "nc":    8,
    "names": {
        0: "person",   1: "bicycle", 2: "car",   3: "van",
        4: "truck",    5: "tricycle",6: "bus",   7: "motor"
    }
}
yaml_path = WORK_DIR / "visdrone.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f"✅ YAML created at {yaml_path}")

# ---- 4. W&B Setup ---------------------------------------------------------
from kaggle_secrets import UserSecretsClient
try:
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    # Explicitly set the project to ensure YOLO logs to the right place
    os.environ["WANDB_PROJECT"] = "visdrone-det"
    print("✅ W&B API key loaded and project set")
except Exception:
    print("⚠️ W&B key not found. Continuing without W&B.")

from ultralytics import settings
settings.update({"wandb": True})
print("✅ Ultralytics W&B integration enabled")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


✅ train frames: 24,201
✅ val frames:   2,846
✅ YAML created at /kaggle/working/visdrone.yaml


✅ W&B API key loaded and project set
✅ Ultralytics W&B integration enabled


In [2]:
# Optional: 30-second smoke test — runs 1 epoch on 20 images to catch OOM
# Comment this out or delete after success
smoke_model = YOLO("yolov8l.pt")
smoke_model.train(
    data=str(yaml_path),
    epochs=1,
    imgsz=800,
    batch=-1,        # test auto-batch
    device=0,
    fraction=0.001,  # only ~24 images
    project="smoke_test",
    name="smoke",
    plots=False,
    verbose=True,
)
print("✅ Smoke test passed — batch size feasible, starting real training")

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/visdrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.001, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=smoke, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=False, pose=12.0

Overriding model.yaml nc=80 with nc=8



                   from  n    params  module                                       arguments                     


  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 


  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               


... (516 outputs left out)

wandb: 
wandb: Run history:
wandb:                epoch ▁█▁
wandb:               lr/pg0 ▁
wandb:               lr/pg1 ▁
wandb:               lr/pg2 ▁
wandb:     metrics/mAP50(B) █▁
wandb:  metrics/mAP50-95(B) ▁█
wandb: metrics/precision(B) ▁█
wandb:    metrics/recall(B) █▁
wandb:         model/GFLOPs ▁
wandb:     model/parameters ▁
wandb:                   +7 ...
wandb: 
wandb: Run summary:
wandb:                epoch 1
wandb:               lr/pg0 0.00083
wandb:               lr/pg1 0.00083
wandb:               lr/pg2 0.00083
wandb:     metrics/mAP50(B) 0.20688
wandb:  metrics/mAP50-95(B) 0.10501
wandb: metrics/precision(B) 0.38873
wandb:    metrics/recall(B) 0.21185
wandb:         model/GFLOPs 258.477
wandb:     model/parameters 43636008
wandb:                   +7 ...
wandb: 


wandb: 🚀 View run smoke at: https://wandb.ai/suchok-pn-ukrainian-catholic-university/smoke_test/runs/smoke_20261108_185742
wandb: ⭐️ View project at: https://wandb.ai/suchok-pn-ukrainian-catholic-university/smoke_test
wandb: Synced 5 W&B file(s), 0 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./runs/detect/smoke_test/smoke/wandb/run-20261108_185743-smoke_20261108_185742/logs


✅ Smoke test passed — batch size feasible, starting real training


In [3]:
# =============================================================================
# Cell 2: Train YOLOv8l
# =============================================================================
# Configuration:
#   - Model:        YOLOv8l (COCO-pretrained from ultralytics)
#   - Epochs:       48
#   - Image size:   800×800 (higher than default 640 for small objects)
#   - Batch size:   -1 (Ultralytics AutoBatch — picks max feasible on T4)
#   - Optimizer:    SGD, lr0=0.01, cosine schedule, warmup=3 epochs
#   - save_period:  4 (saves a checkpoint every 4 epochs — crash-safe)
#   - W&B:          Logs to `visdrone-det` project as run `yolov8l_run1`
#
# Runtime estimate: 6–9 hours on T4 (48 epochs, imgsz=800)
# If Kaggle's 12h session limit is hit, checkpoints at epochs 4, 8, ..., 44
# are safe. Resume from `last.pt` in a new session if needed.
# =============================================================================

model = YOLO("yolov8l.pt")   # loads COCO-pretrained weights from internet

results = model.train(
    data=str(yaml_path),
    epochs=48,
    imgsz=800,
    batch=-1,                # AutoBatch
    device=0,                # T4 GPU
    project="visdrone_yolo",
    name="yolov8l_800",
    save_period=4,
    # Optimizer
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    # Reproducibility
    seed=28,
    deterministic=True,
    # Output
    plots=True,
    verbose=True,
)

print("✅ Training finished!")

# Copy best weights to a clean output folder
best_weights = Path("visdrone_yolo/yolov8l_800/weights/best.pt")
if best_weights.exists():
    shutil.copy(best_weights, OUTPUT_DIR / "yolov8l_best.pt")
    print(f"✅ Best weights copied to {OUTPUT_DIR / 'yolov8l_best.pt'}")
else:
    print("⚠️ Best weights not found. Check training logs.")

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/visdrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=48, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8l_800, nbs=64, nms=None, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12

Overriding model.yaml nc=80 with nc=8



                   from  n    params  module                                       arguments                     


  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 


  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               


... (95,976 outputs left out)

      26/48      7.96G     0.8845     0.4032      0.834        756        800: 91% ━━━━━━━━━━╸─ 3166/3458 2.2it/s 23:44<2:11

      26/48      7.96G     0.8845     0.4032      0.834        600        800: 91% ━━━━━━━━━━╸─ 3167/3458 2.2it/s 23:44<2:11

      26/48      7.96G     0.8845     0.4032     0.8341        459        800: 91% ━━━━━━━━━━╸─ 3168/3458 2.2it/s 23:45<2:10

      26/48      7.96G     0.8846     0.4033     0.8341        590        800: 91% ━━━━━━━━━━╸─ 3169/3458 2.2it/s 23:45<2:09

In [ ]:
# =============================================================================
# Cell 3: Final Validation on Full VisDrone val set (2,846 frames)
# =============================================================================
# This cell loads `best.pt` and evaluates it on the full validation split using
# Ultralytics' built-in COCO-style metrics:
#   - mAP50:    mean AP at IoU=0.50 (comparable to Kaggle LB)
#   - mAP50-95: mean AP averaged over IoU=0.50:0.95 (COCO primary metric)
#   - per-class AP: list of 8 values (one per VisDrone class)
#
# Runtime estimate: ~15 min on T4
# =============================================================================

from ultralytics import YOLO

best_weights = OUTPUT_DIR / "yolov8l_best.pt"
model = YOLO(str(best_weights))

metrics = model.val(
    data=str(yaml_path),
    imgsz=800,
    batch=16,
    device=0,
    split="val",
    project="visdrone_yolo",
    name="yolov8l_val",
)

print("=" * 60)
print("YOLOv8l — Final Validation Results")
print("=" * 60)
print(f"  mAP50:    {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Per-class mAP50: {metrics.box.maps}")

In [ ]:
# =============================================================================
# Cell 4: Package All Artifacts for Download
# =============================================================================
# Copies the important files (weights, results, curves) into `OUTPUT_DIR`
# so you can download them as a single Kaggle dataset for Notebook B.
# =============================================================================

artifacts = [
    "visdrone_yolo/yolov8l_800/weights/best.pt",
    "visdrone_yolo/yolov8l_800/weights/last.pt",
    "visdrone_yolo/yolov8l_800/results.csv",
    "visdrone_yolo/yolov8l_800/confusion_matrix.png",
    "visdrone_yolo/yolov8l_800/PR_curve.png",
    "visdrone_yolo/yolov8l_800/F1_curve.png",
]

for f in artifacts:
    src = Path(f)
    if src.exists():
        shutil.copy(src, OUTPUT_DIR / src.name)
        print(f"✅ Copied {src.name}")
    else:
        print(f"⚠️ Missing: {f}")

print("\n📦 Artifacts in", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("*")):
    print(f"   {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")

print("\n🎯 Next step: save this output as a Kaggle dataset (e.g., `yolo-l-v1`)")
print("   Then proceed to Notebook B (SAHI inference + submission).")

## Troubleshooting

### If AutoBatch fails (OOM error)
Change `batch=-1` in Cell 2 to `batch=8` and rerun Cell 2 only.

### If Kaggle hits the 12-hour limit during training
The output still contains checkpoints at epochs 4, 8, 12, ..., 44. In a new session:
```python
from ultralytics import YOLO
model = YOLO("path/to/last.pt")
model.train(resume=True)   # finishes the original 48-epoch plan


---

## 📋 What to paste in the notebook (final checklist)

| # | Type | Content |
|---|---|---|
| 1 | Markdown | Notebook Title |
| 2 | Code | Setup & YAML |
| 3 | Code | Training |
| 4 | Code | Final Validation |
| 5 | Code | Package Artifacts |
| 6 | Markdown | Troubleshooting (optional) |

That's **6 cells total** — 4 code, 2 markdown. Copy them in order.

---

## 🚀 How to run

1. **Create a new Kaggle notebook** with **Internet ON** and **T4 GPU**.
2. **Attach `suchok/my-dataset`.**
3. **Paste the 6 cells above** in order.
4. **Click Save Version → Save & Run All (Commit).**
5. **Watch W&B** (`visdrone-det` → `yolov8l_run1`) to monitor loss in real time from any browser — you don't need to sit in front of the Kaggle console.
6. **When done**, save the notebook output as a Kaggle dataset (`yolo-l-v1`).

Let me know when training completes — paste the `metrics.box.map50` line from Cell 3, and I'll write **Notebook B (SAHI inference + tuning + submission generation)**.